In [1]:
"""
AI Air Pencil - Hand Gesture Based Drawing System (Single File Version)

Gestures:
    Index finger only up      -> DRAW
    Index + Middle up         -> SELECT (pick a color from the toolbar)
    Open palm (all 5 up)      -> ERASE
    Fist (all down)           -> IDLE

Keyboard:
    c - clear canvas
    q - quit

Requires: opencv-python, mediapipe, numpy
    pip install opencv-python mediapipe numpy
"""

import math
import cv2
import numpy as np
import mediapipe as mp


# ============================================================
# CONFIG
# ============================================================

CAMERA_INDEX = 0

# MediaPipe
MAX_HANDS = 1
MIN_DETECTION_CONFIDENCE = 0.7
MIN_TRACKING_CONFIDENCE = 0.7

# Drawing
BRUSH_THICKNESS = 5
ERASER_THICKNESS = 80

# OpenCV uses BGR
GREEN = (0, 255, 0)
BLUE = (255, 0, 0)
RED = (0, 0, 255)
YELLOW = (0, 255, 255)
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)

CURRENT_COLOR = GREEN

# Smoothing
SMOOTHING_ALPHA = 0.4

# Gesture Stabilization
GESTURE_FRAMES_REQUIRED = 4

# Jump Rejection
MAX_JUMP_DISTANCE = 80

# Toolbar
TOOLBAR_HEIGHT = 80

COLOR_BUTTONS = [
    ("GREEN", GREEN, (20, 10, 120, 65)),
    ("BLUE", BLUE, (140, 10, 240, 65)),
    ("RED", RED, (260, 10, 360, 65)),
    ("YELLOW", YELLOW, (380, 10, 500, 65)),
]

TIP_IDS = [4, 8, 12, 16, 20]


# ============================================================
# UTILS - smoothing + distance
# ============================================================

class CoordinateSmoother:
    def __init__(self):
        self.x = None
        self.y = None

    def smooth(self, raw_x, raw_y):
        alpha = SMOOTHING_ALPHA

        if self.x is None or self.y is None:
            self.x = raw_x
            self.y = raw_y
        else:
            self.x = alpha * raw_x + (1 - alpha) * self.x
            self.y = alpha * raw_y + (1 - alpha) * self.y

        return int(self.x), int(self.y)

    def reset(self):
        self.x = None
        self.y = None


def point_distance(point1, point2):
    return math.hypot(point1[0] - point2[0], point1[1] - point2[1])


# ============================================================
# HAND TRACKER
# ============================================================

class HandTracker:
    def __init__(self):
        self.mp_hands = mp.solutions.hands
        self.mp_draw = mp.solutions.drawing_utils

        self.hands = self.mp_hands.Hands(
            max_num_hands=MAX_HANDS,
            min_detection_confidence=MIN_DETECTION_CONFIDENCE,
            min_tracking_confidence=MIN_TRACKING_CONFIDENCE,
        )

    def detect(self, frame):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.hands.process(rgb)

        if not results.multi_hand_landmarks:
            return None, None

        hand = results.multi_hand_landmarks[0]

        # Draw skeleton
        self.mp_draw.draw_landmarks(frame, hand, self.mp_hands.HAND_CONNECTIONS)

        h, w, _ = frame.shape
        landmarks = []
        for lm in hand.landmark:
            x = int(lm.x * w)
            y = int(lm.y * h)
            landmarks.append((x, y))

        hand_type = results.multi_handedness[0].classification[0].label

        return landmarks, hand_type

    def close(self):
        self.hands.close()


# ============================================================
# GESTURE DETECTION + STABILIZATION
# ============================================================

def get_finger_states(landmarks, hand_type):
    fingers = []

    # Thumb
    if hand_type == "Right":
        fingers.append(1 if landmarks[4][0] < landmarks[3][0] else 0)
    else:
        fingers.append(1 if landmarks[4][0] > landmarks[3][0] else 0)

    # Index, Middle, Ring, Little
    for tip in TIP_IDS[1:]:
        fingers.append(1 if landmarks[tip][1] < landmarks[tip - 2][1] else 0)

    return fingers


def detect_gesture(fingers):
    if fingers == [0, 1, 0, 0, 0]:
        return "DRAW"
    elif fingers == [0, 1, 1, 0, 0]:
        return "SELECT"
    elif fingers == [1, 1, 1, 1, 1]:
        return "ERASE"
    elif fingers == [0, 0, 0, 0, 0]:
        return "IDLE"
    return "IDLE"


class GestureStabilizer:
    def __init__(self):
        self.candidate = None
        self.count = 0
        self.stable_gesture = "IDLE"

    def update(self, gesture):
        if gesture == self.candidate:
            self.count += 1
        else:
            self.candidate = gesture
            self.count = 1

        if self.count >= GESTURE_FRAMES_REQUIRED:
            self.stable_gesture = gesture

        return self.stable_gesture


# ============================================================
# PAINTER - drawing / erasing / combining canvas with camera
# ============================================================

class Painter:
    def __init__(self):
        self.canvas = None
        self.previous_point = None

    def create_canvas(self, frame):
        if self.canvas is None:
            self.canvas = np.zeros_like(frame)

    def draw(self, current_point, color):
        if self.previous_point is None:
            self.previous_point = current_point
            return

        distance = point_distance(self.previous_point, current_point)

        # Reject suspicious jumps
        if distance <= MAX_JUMP_DISTANCE:
            cv2.line(self.canvas, self.previous_point, current_point, color, BRUSH_THICKNESS)

        self.previous_point = current_point

    def erase(self, point):
        cv2.circle(self.canvas, point, ERASER_THICKNESS // 2, (0, 0, 0), -1)

    def reset_previous(self):
        self.previous_point = None

    def clear(self):
        if self.canvas is not None:
            self.canvas[:] = 0
        self.previous_point = None

    def create_mask(self):
        gray_canvas = cv2.cvtColor(self.canvas, cv2.COLOR_BGR2GRAY)
        _, drawing_mask = cv2.threshold(gray_canvas, 1, 255, cv2.THRESH_BINARY)
        return drawing_mask

    def combine(self, frame):
        drawing_mask = self.create_mask()
        inverse_mask = cv2.bitwise_not(drawing_mask)

        frame_background = cv2.bitwise_and(frame, frame, mask=inverse_mask)
        canvas_foreground = cv2.bitwise_and(self.canvas, self.canvas, mask=drawing_mask)

        return cv2.add(frame_background, canvas_foreground)


# ============================================================
# UI - toolbar, cursor, text overlays
# ============================================================

def draw_toolbar(frame, current_color):
    cv2.rectangle(frame, (0, 0), (frame.shape[1], TOOLBAR_HEIGHT), (40, 40, 40), -1)

    for name, color, box in COLOR_BUTTONS:
        x1, y1, x2, y2 = box

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, -1)
        cv2.putText(frame, name, (x1 + 10, y1 + 35), cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 2)

        if color == current_color:
            cv2.rectangle(frame, (x1 - 3, y1 - 3), (x2 + 3, y2 + 3), WHITE, 3)


def select_color(point, current_color):
    x, y = point

    for name, color, box in COLOR_BUTTONS:
        x1, y1, x2, y2 = box
        if x1 <= x <= x2 and y1 <= y <= y2:
            return color

    return current_color


def draw_mode(frame, mode):
    cv2.putText(frame, mode, (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, WHITE, 2)


def draw_cursor(frame, point, color):
    cv2.circle(frame, point, 8, color, -1)


def draw_eraser_cursor(frame, point):
    cv2.circle(frame, point, ERASER_THICKNESS // 2, WHITE, 2)


def draw_coordinates(frame, point):
    x, y = point
    cv2.putText(frame, f"({x}, {y})", (x + 10, y + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 2)


# ============================================================
# MAIN LOOP
# ============================================================

def main():
    cap = cv2.VideoCapture(CAMERA_INDEX)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    hand_tracker = HandTracker()
    gesture_stabilizer = GestureStabilizer()
    smoother = CoordinateSmoother()
    painter = Painter()

    current_color = CURRENT_COLOR

    while True:
        success, frame = cap.read()
        if not success:
            print("Failed to read camera.")
            break

        # Mirror webcam
        frame = cv2.flip(frame, 1)

        # Create canvas (once, based on frame size)
        painter.create_canvas(frame)

        # Detect hand
        landmarks, hand_type = hand_tracker.detect(frame)

        # Draw toolbar (base layer, before hand overlays)
        draw_toolbar(frame, current_color)

        if landmarks is not None:
            fingers = get_finger_states(landmarks, hand_type)

            raw_gesture = detect_gesture(fingers)
            stable_gesture = gesture_stabilizer.update(raw_gesture)

            # Index fingertip (landmark 8)
            raw_x, raw_y = landmarks[8]
            x, y = smoother.smooth(raw_x, raw_y)
            current_point = (x, y)

            draw_mode(frame, stable_gesture)
            draw_coordinates(frame, current_point)

            if stable_gesture == "DRAW":
                draw_cursor(frame, current_point, current_color)

                # Don't draw over toolbar
                if y > TOOLBAR_HEIGHT:
                    painter.draw(current_point, current_color)
                else:
                    painter.reset_previous()

            elif stable_gesture == "SELECT":
                painter.reset_previous()
                current_color = select_color(current_point, current_color)

            elif stable_gesture == "ERASE":
                painter.reset_previous()
                draw_eraser_cursor(frame, current_point)

                if y > TOOLBAR_HEIGHT:
                    painter.erase(current_point)

            else:  # IDLE
                painter.reset_previous()

        else:
            painter.reset_previous()
            smoother.reset()

        # Redraw toolbar on top so the color highlight stays visible
        draw_toolbar(frame, current_color)

        # Combine webcam feed + drawing canvas
        output = painter.combine(frame)

        cv2.imshow("AI Air Pencil", output)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('c'):
            painter.clear()
        elif key == ord('q'):
            break

    cap.release()
    hand_tracker.close()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()